# TLA-Prover team lab

This is the shared inspection surface for the prover program. It answers three different questions without blending them:

1. Does the TLA+ toolchain work?
2. What is reachable and running on the research backends?
3. Did a candidate improve the frozen protected prover gate?

Only the third question can promote a model. This notebook is deliberately read-only with respect to clusters: it never submits, cancels, or promotes a job.

In [ ]:
from dataclasses import asdict
import json
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'tools').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import tlakit
from tlakit.remote import RemoteRunner
from tools.tlakit_cluster_backend import default_backends

print('TLAKit', tlakit.__version__)
print('Repository root:', ROOT)

## What a training run means

A run is a bounded experiment, not merely a process that consumed GPU time. The team freezes its data and acceptance rule, validates the environment, trains a checkpoint, evaluates it with the registered verifiers, audits the meaning of each pass, and only then considers promotion.

Lower loss, more samples, a healthy login, a completed scheduler job, or a SANY-only parse pass are useful observations. None is a prover result. The protected target is currently **0/119** Stage 3 TLAPS cases.

In [ ]:
run_contract = [
    {'stage': 'freeze', 'question': 'Are data, model, budget, and acceptance rule pinned?'},
    {'stage': 'preflight', 'question': 'Are dependencies, hashes, output paths, and trainable parameters valid?'},
    {'stage': 'train', 'question': 'Did the bounded optimizer produce the registered checkpoint and append-only log?'},
    {'stage': 'evaluate', 'question': 'Did the same SANY/TLC/TLAPS checks run at the registered budget?'},
    {'stage': 'audit', 'question': 'Were vacuous and semantically weakened candidates rejected?'},
    {'stage': 'promote', 'question': 'Did the protected gate improve strictly with genuine proof evidence?'},
]
run_contract

## Model × method matrix

A model name is not a training method. LoRA/PEFT learns small adapters while the base weights remain frozen; full fine-tuning updates the selected base weights; prompt-only runs update no parameters. This matrix is intentionally explicit about failed, retracted, shelved, and not-run work.

In [ ]:
model_method_matrix = [
    {'base_model': 'gpt-oss-20b', 'method': 'Prompt-only', 'update_scope': 'No parameters', 'run': 'Gate-2 baseline', 'outcome': 'Measured comparison arm', 'status': 'baseline'},
    {'base_model': 'gpt-oss-120b', 'method': 'Prompt-only', 'update_scope': 'No parameters', 'run': 'Gate-2 baseline', 'outcome': 'A 12/30; B 21/23 pass@32', 'status': 'baseline'},
    {'base_model': 'gpt-oss-20b', 'method': 'LoRA / PEFT SFT', 'update_scope': 'Expert-reaching adapters', 'run': 'v2_sft1 · 39 plain-text pairs', 'outcome': '0/10; 82% unextractable', 'status': 'failed'},
    {'base_model': 'gpt-oss-120b', 'method': 'LoRA / PEFT SFT', 'update_scope': 'Harmony-formatted adapters', 'run': 'v2_sft2 · 260 pairs', 'outcome': 'A 11/30; B 18/23', 'status': 'failed'},
    {'base_model': 'gpt-oss-20b', 'method': 'LoRA / PEFT SFT', 'update_scope': 'Expert-LoRA', 'run': 'W2.6 repair-v1 · 508 pairs', 'outcome': '+1 pass@1; −8 pass@4', 'status': 'shelved'},
    {'base_model': 'Qwen3.6-27B', 'method': 'Intended LoRA / PEFT SFT', 'update_scope': 'Resolver matched nothing', 'run': 'Retracted comparison', 'outcome': '0.0195% trainable', 'status': 'retracted'},
    {'base_model': 'Any', 'method': 'Full fine-tuning', 'update_scope': 'All selected base weights', 'run': 'No registered run', 'outcome': 'Not done', 'status': 'not_run'},
    {'base_model': 'Any', 'method': 'Stage-3 RL / GRPO', 'update_scope': 'Policy update from verified reward', 'run': 'No protected result', 'outcome': 'Not done; target 0/119', 'status': 'not_run'},
]
model_method_matrix

## Ground-truth toolchain smoke check

This exercises TLAKit and SANY through the public runner. A pass verifies the lab path and the toolchain; it does not score a model or change the protected gate.

In [ ]:
SOURCE = r'''---- MODULE NotebookSmoke ----
EXTENDS Naturals
THEOREM OnePlusOne == 1 + 1 = 2
===='''
runner = RemoteRunner()
health = runner.health()
parsed = runner.parse(SOURCE, 'NotebookSmoke')
toolchain_smoke = {
    'service': health,
    'outcome': parsed.outcome.value,
    'diagnostics': len(parsed.diagnostics),
    'claim_boundary': 'toolchain only',
}
toolchain_smoke

## Evidence ledger: what is actually present here

This scan is local and read-only. It lists recent run directories and the evidence files they contain; it does not infer a score from a folder name. Open the linked `config.json`, rows, summary, and audit artifacts together before quoting a result.

In [ ]:
def evidence_index(root: Path, limit: int = 12):
    runs_root = root / 'results' / 'runs'
    if not runs_root.is_dir():
        return []
    entries = []
    for run_dir in sorted((p for p in runs_root.iterdir() if p.is_dir()), key=lambda p: p.stat().st_mtime, reverse=True):
        files = {p.name for p in run_dir.iterdir() if p.is_file()}
        entries.append({
            'run_id': run_dir.name,
            'has_config': 'config.json' in files,
            'has_rows': 'rows.jsonl' in files,
            'has_summary': any(name.startswith('summary') for name in files),
            'file_count': len(files),
        })
        if len(entries) >= limit:
            break
    return entries

recent_evidence = evidence_index(ROOT)
recent_evidence if recent_evidence else {'note': 'No results/runs directory in this notebook checkout.'}

## Polaris, Sophia, and current user-owned processes

The probes are read-only and non-interactive. They use SSH batch mode, so expired authentication becomes an honest unavailable status instead of a hidden prompt. The process view reports only the current user's PID, state, CPU/memory counters, elapsed time, and a coarse command purpose; it never returns argv strings, environment variables, logs, or scheduler controls.

In [ ]:
cluster_view = []
for backend in default_backends():
    status = asdict(backend.probe())
    status['processes'] = [asdict(process) for process in backend.processes()]
    status['control_boundary'] = 'read-only; no scheduler submission or cancellation'
    cluster_view.append(status)

cluster_view

## Published team snapshot

The research hub carries a safe, reviewable snapshot so teammates can orient themselves without cluster credentials. It is intentionally separate from the fresh probe above.

In [ ]:
status_path = ROOT / 'site' / 'status.json'
published_status = json.loads(status_path.read_text()) if status_path.is_file() else {
    'note': 'This checkout does not include the published site snapshot.'
}
{
    'published_at': published_status.get('published_at'),
    'phase': published_status.get('program', {}).get('phase'),
    'protected_gate': published_status.get('program', {}).get('gate'),
    'freshness_note': published_status.get('freshness_note'),
}

## SkillOpt-derived experiment discipline

We borrow the mechanics, not the metric, from [Microsoft SkillOpt](https://github.com/microsoft/SkillOpt): preserve scored trajectories, propose bounded changes, keep rejected changes as negative evidence, and accept only strict held-out improvement. The protected 119-case gate remains fixed and never becomes training data.

In [ ]:
experiment_policy = {
    'target': 'frozen protected prover checkpoint',
    'optimizer_input': 'non-protected scored trajectories only',
    'update_budget': 'one bounded hypothesis per child artifact',
    'negative_memory': 'retain rejected edits and normalized failure signatures',
    'promotion': 'strict protected-gate improvement with SANY/TLC/TLAPS evidence',
    'forbidden_proxies': ['training loss', 'GPU utilization', 'SSH health', 'TLAKit health', 'optimizer preference'],
    'actions_available_here': 'inspect and probe only; submit/cancel/promote elsewhere',
}
experiment_policy